### Notebook objectives:
    - extract burstiness values from daylong recordings 
    - save and export as csvs for analyses 

## imports & misc things to load

In [1]:
# imports 
import os, math, statistics, ast
import numpy as np
import pandas as pd 
import nltk 

In [2]:
import spacy
nlp = spacy.load("en_core_web_sm")

In [3]:
# set directory 
os.chdir(os.path.expanduser("/Users/zeynepmarasli/Downloads/Documents/Projects/Daylong_Analyses/Python_Code"))
# import daylongtranscript module -- can be found under "Simulation and Evaluation of Sampling Methods for Daylong Audio Data" repo on GitHub
from module_lib.daylongtranscript import*

loaded


In [4]:
## load daylong transcripts
# this is done via creating a DaylongTranscript object for each transcript 
# requires a tab-delimited file of the raw transcript chat file (can be done using cleantranscript.py under "Simulation and Evaluation of Sampling Methods for Daylong Audio Data" repo on GitHub)
transcriptA1 = DaylongTranscript(fpath = "Transcripts/For_OSF/A787_001107_cleaned.txt", fname = "A787_001107",isVanDam=False)
transcriptA2 = DaylongTranscript(fpath = "Transcripts/For_OSF/A787_001109_cleaned.txt", fname = "A787_001109", isVanDam=False)
transcriptA3 = DaylongTranscript(fpath = "Transcripts/For_OSF/A787_001111_cleaned.txt", fname = "A787_001109", isVanDam=False)
transcriptB1 = DaylongTranscript(fpath = "Transcripts/For_OSF/B895_010002_cleaned.txt", fname = "B895_010002",isVanDam=False)
transcriptB2 = DaylongTranscript(fpath = "Transcripts/For_OSF/B895_010004_cleaned.txt", fname = "B895_010004", isVanDam=False)

transcripts = [transcriptA1, transcriptA2, transcriptA3, transcriptB1, transcriptB2]

In [15]:
# FUNCTIONS
def find_events(target_word, df, use_adjusted_timestamps = False):
    '''
    - target  = a word 
    - returns list of event onset times (start timestamps of utterance in which word appears)
    - if a word occurs more than once in a given utterance, the same timestamp value will be included multiple times
    '''
    event_indices = []
    for i, utt in enumerate(df['cleaned_utt']):
        tokens = utt.split()
        events = tokens.count(target_word)
        if events > 0: 
            for e in range(events): event_indices.append(i)
    if use_adjusted_timestamps: event_onsets = list(df.loc[event_indices]["start_ts_adjusted"].values)
    else: event_onsets = list(df.loc[event_indices]["start_ts"].values)
    return event_onsets

def get_interevent_intervals(target_word, df, use_adjusted_timestamps = False):
    '''
    - target  = a word 
    - for each pair of event times, takes time difference (T2 - T1) = interevent interval
    - returns list of IEIs for each pair of events 
    '''
    event_onsets = find_events(target_word, df, use_adjusted_timestamps = use_adjusted_timestamps)
    event_onsets = [int(x) for x in event_onsets]
    interevent_intervals = []
    for i, event in enumerate(event_onsets):
        if i == len(event_onsets) - 1: break
        else:
         iei = event_onsets[i+1] - event
         interevent_intervals.append(iei)
    #print("\tIEIS: ", interevent_intervals)
    return interevent_intervals, event_onsets

def calc_burstiness(ieis, alternative = False ):
    ''' 
    - given a list of ieis, calculates a burstiness value 
    - by defaul will calculate using Goh & Barabasi (2007) formula 
    - alternative is for Kim & Jo (2016) alt formula 
    '''
    if len(ieis) < 1: # need at least 2 ieis to calculuate mean and stdev values 
        return np.nan
    elif statistics.mean(ieis) == 0: return np.nan
    else: 
        r = statistics.pstdev(ieis) / statistics.mean(ieis)
        if alternative is False:
            if r+1 == 0: return np.nan
            else: return (r-1)/(r+1)
        else: #(alternative is True)
            n = len(ieis) + 1 #num of events = num of ieis + 1 
            num = (math.sqrt(n+1)* r) - math.sqrt(n-1)
            denom = ( (math.sqrt(n+1)-2)*r) + math.sqrt(n-1)
            return num/denom
        
def get_all_terms(df):
    '''  
    - given dataframe of transcript, will return all transcribed speech as a list of all word forms  
    '''
    word_list = []
    for words in df['cleaned_utt']:
        #print(words)
        word_list.extend(words.split())
    return word_list 
       
def get_b_values(df, ieis = False, alternative=False, use_adjusted_timestamps = False ):
    '''  
    - wrapper function to get all b values for all words from for a given transcript 
    '''
    all_tokens = get_all_terms(df)
    unique_terms = set(all_tokens)
    burstiness_values = {term: None for term in unique_terms}
    ieis = {term: None for term in unique_terms}
    occurrences = {term: None for term in unique_terms}
    for target_word in burstiness_values.keys(): 
        word_ieis, word_occurrences = get_interevent_intervals(target_word, df, use_adjusted_timestamps = use_adjusted_timestamps)
        b_value = calc_burstiness(word_ieis, alternative=alternative)
        burstiness_values[target_word] = b_value
        ieis[target_word] = word_ieis
        occurrences[target_word] = word_occurrences
    if ieis: return burstiness_values, ieis, occurrences
    else: return burstiness_values 

def get_frequencies(df, words):
    allspeech = get_all_terms(df)
    frequencies = {term: None for term in words}
    for word in frequencies.keys(): frequencies[word] = allspeech.count(word)
    return frequencies 

def get_pos_tags(words):
    pos_tags = []
    for word in words:
        doc = nlp(word)
        for token in doc[0:1]: pos_tags.append(token.pos_)
    return pos_tags

In [6]:
# read in transcript dataframes
df_A1 = pd.read_csv("Transcripts/For_OSF/A787_001107_cleaned.txt", sep = "\t", header=None)
df_A1 = df_A1.rename(columns={0: 'speaker', 1: 'cleaned_utt', 2: 'raw_utt', 3: 'start_ts', 4: 'stop_ts', 5: 'xds', 6: 'subtiers'})

df_A2 = pd.read_csv("Transcripts/For_OSF/A787_001109_cleaned.txt", sep = "\t", header=None)
df_A2 = df_A2.rename(columns={0: 'speaker', 1: 'cleaned_utt', 2: 'raw_utt', 3: 'start_ts', 4: 'stop_ts', 5: 'xds', 6: 'subtiers'})

df_A3 = pd.read_csv("Transcripts/For_OSF/A787_001111_cleaned.txt", sep = "\t", header=None)
df_A3 = df_A3.rename(columns={0: 'speaker', 1: 'cleaned_utt', 2: 'raw_utt', 3: 'start_ts', 4: 'stop_ts', 5: 'xds', 6: 'subtiers'})

df_B1 = pd.read_csv("Transcripts/For_OSF/B895_010002_cleaned.txt", sep = "\t", header=None)
df_B1 = df_B1.rename(columns={0: 'speaker', 1: 'cleaned_utt', 2: 'raw_utt', 3: 'start_ts', 4: 'stop_ts', 5: 'xds', 6: 'subtiers'})

df_B2 = pd.read_csv("Transcripts/For_OSF/B895_010004_cleaned.txt", sep = "\t", header=None)
df_B2 = df_B2.rename(columns={0: 'speaker', 1: 'cleaned_utt', 2: 'raw_utt', 3: 'start_ts', 4: 'stop_ts', 5: 'xds', 6: 'subtiers'})

transcript_dfs = [df_A1, df_A2, df_A3, df_B1,df_B2]
transcript_labels = ["A1", "A2", "A3","B1", "B2"]

## Get B values from daylong transcripts for all words

### 1. make dataframe with adjust timestamps --> removes silences > 30 min in transcript & adjusts timestamps of utterances 

In [7]:
transcript_dfs_adjusted = []
for i, df_transcript in enumerate(transcript_dfs):
    transcript = transcripts[i] # get daylong transcript
    silence_pairs = transcript.get_silence_intervals()[0]
    df = df_transcript.copy()
    # create new columns for adjusted timestamps
    df["start_ts_adjusted"] = df["start_ts"]
    df["stop_ts_adjusted"] = df["stop_ts"]
    for sp in silence_pairs: 
        sp_start = sp[0]
        sp_end = sp[1]
        silence_length = sp_end - sp_start
        # rows in df that need to be adjusted 
        condition = df['start_ts'] >= sp_end
        # Change values in start_ts and stop_ts where condition is True
        df.loc[condition, ['start_ts_adjusted', 'stop_ts_adjusted']] =  df.loc[condition, ['start_ts_adjusted', 'stop_ts_adjusted']] - silence_length
    transcript_dfs_adjusted.append(df)

In [8]:
df = transcript_dfs_adjusted[0]
df = df[df["cleaned_utt"] != '0']
df = df.dropna(subset=['cleaned_utt', 'start_ts', 'stop_ts']).reset_index()
df

,index,speaker,cleaned_utt,raw_utt,start_ts,stop_ts,xds,subtiers,start_ts_adjusted,stop_ts_adjusted
0,0,FA1,good morning,good morning.,25480,27970,T,NaN,25480,27970
1,1,FA1,morning,morning.,34360,36270,T,NaN,34360,36270
2,2,FA1,hi,hi.,37460,37845,T,NaN,37460,37845
3,3,FA1,hi,hi.,38920,39200,T,NaN,38920,39200
4,4,FA1,you gettin' up,you <gettin'> [: getting] up?,43450,44530,T,NaN,43450,44530
...,...,...,...,...,...,...,...,...,...,...
3594,5434,FA1,I love you,I love you [ &= kiss].,45359180,45361208,T,NaN,11230202,11232230
3595,5436,FA1,good night,<good night> [ &= whispers].,45362660,45363398,T,NaN,11233682,11234420
3596,5437,FA1,&= kiss sounds,&= kiss sounds.,45363994,45365204,T,NaN,11235016,11236226
3597,5439,FA1,good night,good night.,45367202,45367834,T,NaN,11238224,11238856


### 2. get burstiness values from each transcript 

In [ ]:
bursty_dfs_adjusted = []
for i, transcript in enumerate(transcript_labels):
    print(transcript)
    df = transcript_dfs_adjusted[i]
    df = df[df["cleaned_utt"] != '0']
    df = df.dropna(how = "all").reset_index()#df.dropna(subset=['cleaned_utt', 'start_ts', 'stop_ts']).reset_index()
    burstiness_values, ieis, event_onsets = get_b_values(df, ieis=True, alternative=True, use_adjusted_timestamps = True)
    frequencies = get_frequencies(df, burstiness_values.keys())
    pos_tags = get_pos_tags(list(burstiness_values.keys()))
    transcript_label = [transcript] * len(burstiness_values)
    df = pd.DataFrame({"Word": burstiness_values.keys(), "Burstiness": burstiness_values.values(), 
                       "Frequency": frequencies.values(), "Interevent Intervals": ieis.values(), 
                       "Event Onsets": event_onsets.values(), "POS Tag": pos_tags, "Transcript": transcript_label})
    bursty_dfs_adjusted.append(df)

In [18]:
test_df = pd.concat(bursty_dfs_adjusted, ignore_index=True)
test_df.head()

,Word,Burstiness,Frequency,Interevent Intervals,Event Onsets,POS Tag,Transcript
0,asleep,0.311664,5,"[1710, 6380, 2060, 28120]","[1997420, 1999130, 2005510, 2007570, 2035690]",ADJ,A1
1,why'd,NaN,1,[],[6890935],SCONJ,A1
2,dresses,NaN,1,[],[3577742],NOUN,A1
3,holding,NaN,1,[],[6217774],VERB,A1
4,like,0.323744,99,"[22065, 20670, 1760, 4060, 9155, 15635, 69050,...","[671975, 694040, 714710, 716470, 720530, 72968...",INTJ,A1


In [17]:
test_df[test_df["Word"] == "oranges"]

,Word,Burstiness,Frequency,Interevent Intervals,Event Onsets,POS Tag,Transcript
5126,oranges,0.389787,10,"[304122.0, 573673.0, 3437.0, 2300452.0, 12880....","[15990992.0, 16295114.0, 16868787.0, 16872224....",VERB,A3


In [27]:
test_df = test_df.dropna()
test_df = test_df[test_df["Frequency"] > 2]
test_df.describe()

,Burstiness,Frequency
count,5154.000000,5154.000000
mean,0.265331,29.583624
std,0.368460,88.665239
min,-0.999841,3.000000
25%,0.107167,4.000000
50%,0.319558,7.000000
75%,0.477744,18.000000
max,0.992793,1926.000000


In [23]:
og_df = pd.read_csv("Burstiness/ICB_Public/CSVS/b_values_daylong_adjusted_combined.csv", header = 0, sep = "\t" )
og_df['Burstiness'] = og_df['Burstiness'].astype(float)
og_df = og_df.dropna()
og_df = og_df[og_df["Frequency"] > 2]
og_df.describe()

,Burstiness,Frequency
count,5154.000000,5154.000000
mean,0.265258,29.581296
std,0.368544,88.664506
min,-0.999841,3.000000
25%,0.107062,4.000000
50%,0.319534,7.000000
75%,0.477744,18.000000
max,0.992794,1926.000000


In [39]:
test_df.to_csv("Burstiness/ICB_Public/CSVS/b_values_daylong_adjusted_combined_210.csv", sep="\t", index=False, na_rep="NAN")

In [ ]:
combined_df = pd.concat(bursty_dfs_adjusted, ignore_index=True)

# write to csv folder 
#combined_df.to_csv("CSVS/b_values_daylong_adjusted_combined.csv", sep="\t", index=False, na_rep="NAN")
combined_df.head()

,Word,Burstiness,Frequency,Interevent Intervals,Event Onsets,POS Tag,Transcript
0,area,NaN,1,[],[3249026],NOUN,A1
1,wish,-0.494129,3,"[2551668, 5393804]","[1205930, 3757598, 9151402]",VERB,A1
2,fox,NaN,1,[],[1037214],PROPN,A1
3,getchu,NaN,1,[],[7455582],PROPN,A1
4,clear,-1.000000,2,[3212708],"[1022570, 4235278]",ADJ,A1


### 3. make CDI items data from daylong transcripts 

In [37]:
## read in wordbank data 
df_wordbank = pd.read_csv("Burstiness/ICB_Public/CSVS/wordbank_lexicalclasses.csv", header = 0 )
cdi_items = df_wordbank["Term"].to_list()
print(cdi_items)
word_to_category = df_wordbank.set_index('Term')['Lexical_Class'].to_dict()
print(word_to_category)

#word_to_pos = df_childes.set_index('Term')['Part_of_Speech'].to_dict()
#print(word_to_pos)

word_to_wordbank = df_wordbank.set_index('Term')['WordBank'].to_dict()
print(word_to_wordbank)

['alligator', 'animal', 'ant', 'bear', 'bee', 'bird', 'bug', 'bunny', 'butterfly', 'cat', 'chicken', 'cow', 'deer', 'dog', 'donkey', 'duck', 'elephant', 'fish', 'frog', 'giraffe', 'goose', 'hen', 'horse', 'kitty', 'lamb', 'lion', 'monkey', 'moose', 'mouse', 'owl', 'penguin', 'pig', 'pony', 'puppy', 'rooster', 'sheep', 'squirrel', 'teddybear', 'tiger', 'turkey', 'turtle', 'wolf', 'zebra', 'airplane', 'bicycle', 'boat', 'bus', 'car', 'firetruck', 'helicopter', 'motorcycle', 'sled', 'stroller', 'tractor', 'train', 'tricycle', 'truck', 'ball', 'balloon', 'bat', 'block', 'book', 'bubbles', 'chalk', 'crayon', 'doll', 'game', 'glue', 'pen', 'pencil', 'play dough', 'present', 'puzzle', 'story', 'toy', 'apple', 'applesauce', 'banana', 'beans', 'bread', 'butter', 'cake', 'candy', 'carrots', 'cereal', 'cheerios', 'cheese', 'chicken', 'chocolate', 'coffee', 'coke', 'cookie', 'corn', 'cracker', 'donut', 'drink', 'egg', 'fish', 'food', 'french fries', 'grapes', 'green beans', 'gum', 'hamburger', 'ic

In [38]:
def make_daylong_cdi_data(df_data, cdi_items,word_to_category,word_to_wordbank):
    df = df_data
    daylong_mcdi = pd.DataFrame()
    for transcript in transcript_labels:
        df_transcript = df[df["Transcript"] == transcript]
        idx = df_transcript[df_transcript['Word'].isin(cdi_items)].index
        mbcdi = df_transcript.loc[idx]
        daylong_mcdi = pd.concat([daylong_mcdi, mbcdi], ignore_index=True)
    daylong_mcdi = daylong_mcdi[['Word', 'Burstiness', 'Frequency']]
    daylong_mcdi.loc[daylong_mcdi['Frequency'] == 2, 'Burstiness'] = np.nan
    daylong_mcdi = daylong_mcdi.groupby('Word').mean().reset_index()
    daylong_mcdi["Log_Mean_Frequency"] = [math.log(f) for f in daylong_mcdi["Frequency"]]
    daylong_mcdi["Lexical_Class"] = daylong_mcdi['Word'].map(word_to_category)
    daylong_mcdi["WordBank"] = daylong_mcdi['Word'].map(word_to_wordbank)
    daylong_mcdi = daylong_mcdi.rename(columns={'Burstiness': 'Mean_Burstiness', 'Frequency': 'Mean_Frequency'})
    return daylong_mcdi

In [41]:
combined_df = test_df
daylong_cdi = make_daylong_cdi_data(combined_df, cdi_items, word_to_category, word_to_wordbank)
daylong_cdi.to_csv("Burstiness/ICB_Public/CSVS/daylong_adjusted_mcdi_values_0210.csv", index = False, na_rep="NAN")

In [43]:
test_cdi = daylong_cdi
test_cdi.describe()

,Mean_Burstiness,Mean_Frequency,Log_Mean_Frequency,WordBank
count,457.000000,457.000000,457.000000,457.000000
mean,0.328787,41.976769,2.619815,0.414902
std,0.251080,113.615793,1.239984,0.209058
min,-0.580240,3.000000,1.098612,0.040000
25%,0.219670,5.500000,1.704748,0.270000
50%,0.337926,10.000000,2.302585,0.400000
75%,0.469656,22.200000,3.100092,0.580000
max,0.904167,1407.400000,7.249499,0.960000


In [47]:
test_cdi[["Mean_Burstiness", "Log_Mean_Frequency", "Mean_Frequency"]].corr(method='pearson')

,Mean_Burstiness,Log_Mean_Frequency,Mean_Frequency
Mean_Burstiness,1.000000,0.155672,0.079991
Log_Mean_Frequency,0.155672,1.000000,0.717675
Mean_Frequency,0.079991,0.717675,1.000000


In [44]:
og_cdi = pd.read_csv("Burstiness/ICB_Public/CSVS/daylong_adjusted_mcdi_values.csv", header = 0 )
og_cdi.describe()

,Mean_Burstiness,Mean_Frequency,Log_Mean_Frequency,WordBank
count,457.000000,491.000000,491.000000,491.000000
mean,0.328790,38.641412,2.414394,0.414501
std,0.251078,110.212131,1.324131,0.207463
min,-0.580240,2.000000,0.693147,0.040000
25%,0.219670,4.333333,1.466337,0.270000
50%,0.337927,8.000000,2.079442,0.400000
75%,0.469667,19.500000,2.970401,0.570000
max,0.904167,1407.400000,7.249499,0.960000


In [48]:
og_cdi[["Mean_Burstiness", "Log_Mean_Frequency", "Mean_Frequency"]].corr(method='pearson')

,Mean_Burstiness,Log_Mean_Frequency,Mean_Frequency
Mean_Burstiness,1.00000,0.144340,0.077710
Log_Mean_Frequency,0.14434,1.000000,0.694913
Mean_Frequency,0.07771,0.694913,1.000000


# Get B Values from Daylong Sampled 15 min and 5 min
### b values for CDI terms only, averaged across transcripts 

In [ ]:
## imports
# Random Sampler object to randomly sample from daylong transcripts; can be found under 
# "Simulation and Evaluation of Sampling Methods for Daylong Audio Data" repo on GitHub
from module_lib.randomsampler import* 

In [ ]:
interval_size = 5 # size of interval to randomly sample, in minutes; change to 15 for 15 min
total_sampled_time = 500 # how many intervals to sample in total in terms of how many minutes; for 5 min interval, 500 minutes generates 100 5-min samples 
bursty_dfs = []
for i, transcript in enumerate(transcripts):
    print(i)
    df = transcript_dfs[i]
    label = transcript_labels[i]
    print(label)
    sampler = Sampler(transcript = transcript, sampling_interval = 5, total_sampled_time = 500, simulations = 1)
    rts = sampler.generate_random_times()
    for rt in rts:
        start_index, stop_index, first_overlapping, last_overlapping = sampler.find_interval_method4(random_time=rt)
        if start_index is None or stop_index is None: 
            print("nonetype for start or stop index, skipping")
            continue
        utterances = df[start_index:stop_index+1]
        utterances = utterances[['cleaned_utt', 'start_ts', 'stop_ts']]
        utterances = utterances[utterances["cleaned_utt"] != '0']
        utterances.dropna(inplace =True)
        utterances.reset_index(inplace=True, drop = True)
        if len(utterances) == 0: 
            print("no utts")
            continue       
        burstiness_values, ieis, event_onsets = get_b_values(utterances, return_ieis=True, alternative=True)
        frequencies = get_frequencies(utterances, burstiness_values.keys())
        transcript_label = [label] * len(burstiness_values)
        b_df = pd.DataFrame({"Word": burstiness_values.keys(), "Burstiness": burstiness_values.values(), 
                            "Frequency": frequencies.values(), "Interevent Intervals": ieis.values(), 
                            "Event Onsets": event_onsets.values(),  "Transcript": transcript_label})
        bursty_dfs.append(b_df)

In [ ]:
# combine across all intervals 
combined_df_sampled = pd.concat(bursty_dfs, ignore_index=True) 
# group b values by word
df_sampled_grouped = make_daylong_cdi_data(combined_df_sampled, cdi_items,word_to_category, word_to_wordbank)
# export 
df_sampled_grouped.to_csv("CSVS/daylong_sampled_cdi_5minutes.csv", index=False, na_rep="NAN")
# df_sampled_grouped.to_csv("CSVS/daylong_sampled_cdi_15minutes.csv", index=False, na_rep="NAN")

# Get B Values from CHILDES

In [ ]:
# imports 
import tbdb
import pylangacq

In [ ]:
mcdi_items_df = pd.read_csv("CSVS/wordbank_lexicalclasses.csv", index_col=0)
global MCDI_ITEMS 
MCDI_ITEMS = list(mcdi_items_df.index.values)
## get filepath for CHILDES transcripts (on local computer)
childes_fpath = "Documents/Projects/"

In [ ]:
def ms_to_min(x, to_ms = False):
    if to_ms: 
        return x * 60,000 
    else:
        return round(number = (x / 60000),ndigits=2)

def get_events_through_files(files_df, age = "ADD AGE"):
    item_onsets_df = pd.DataFrame()
    files = list(files_df[0].values)
    for file in files: 
        fpath = f"{childes_fpath}{file}"
        print(fpath)
        if not (os.path.exists(fpath)): 
            print("\t transcript cannot be found, moving on")
            continue
        transcript = pylangacq.read_chat(path = fpath)
        utterances_info = transcript.utterances()
        if utterances_info[-1].time_marks is None: 
            print("\tno timestamps, moving on ")
            continue 
        length_ms = utterances_info[-1].time_marks[-1]
        length_min = ms_to_min(length_ms)
        print(f"\t{length_min} minutes")
        if length_min > 31:
            print("\t too long, moving on")
            continue
        utterances_words = transcript.words(by_utterances=True)
        print(f"\t{len(utterances_words)} utterances")

        # find words that occur in mcdi_items and save event onset
        no_timemarks = 0 # if time mark checker didn't catch, this keeps track if there was utterances with no time stamps; if so, do not include in dataframe  
        item_onsets_transcript = {key: [] for key in MCDI_ITEMS}
        for i, utt in enumerate(utterances_words):
            if len(utt) <= 1: continue

            tm = utterances_info[i].time_marks
            if tm is None: 
                print("\t tm is none; ", utt)
                no_timemarks = no_timemarks + 1
                continue 
            utt_onset = tm[0]
            #print(utt, " ", utt_onset)
            
            for item in MCDI_ITEMS:
                occurrences =  utt.count(item)
                if occurrences > 0: 
                    events = [utt_onset] * occurrences
                    item_onsets_transcript[item].extend(events)
        print(f"\t {no_timemarks} missing time marks")           
        if no_timemarks > 20: 
            print("\t insufficient time marks (> 20 missing), moving on")
            continue 

        df_file = pd.DataFrame({ "Word": list(item_onsets_transcript.keys()), "Event_Onsets": list(item_onsets_transcript.values())})
        df_file["Transcript"] = file
        df_file["Age"] = f"{age} months"

        item_onsets_df = pd.concat([item_onsets_df, df_file], ignore_index=True)
    
    return item_onsets_df

def process_df(fpath): 
    df = pd.read_csv(fpath, header=0, sep = "\t")
    df.loc[(df['Event_Onsets']) == '[]', 'Event_Onsets'] = np.nan
    df = df.dropna().reset_index(drop =True)
    df['Event_Onsets'] = df['Event_Onsets'].apply(literal_eval)
    df['Burstiness'] = None
    df["Frequency"] = None
    return df

def burstiness_item_data(df, age = "ADD AGE", mcdi = mcdi_items_df):
    # get averaged item values across transcripts 
    df.loc[df['Frequency'] == 2, 'Burstiness'] = np.nan
    df = df[["Word", "Burstiness", "Frequency", 'Frequency_fromCounter']]
    df_avg = df.groupby(by = "Word").mean()
    df_avg.rename(columns={'Frequency': 'Mean_Frequency'}, inplace=True)
    df_avg.rename(columns={'Frequency_fromCounter': 'Mean_Frequency_fromCounter'}, inplace=True)
    df_avg.rename(columns={'Burstiness': 'Mean_Burstiness'}, inplace=True)
    #df = df.set_index(keys = "Term")
    df = df[["Word", "Frequency", "Frequency_fromCounter"]]
    df_sum = df.groupby(by = 'Word').sum()
    df_sum.rename(columns={'Frequency': 'Sum_Frequency'}, inplace=True)
    df_sum.rename(columns={'Frequency_fromCounter': 'Sum_Frequency_fromCounter'}, inplace=True)
    df_join = df_avg.join(df_sum)
    df_join = df_join[df_join["Mean_Frequency"] != 0]

    # add mcdi lexical class labels 
    df_join = df_join.join(mcdi_items_df, how="left")

    # add age column 
    df_join["Age"] = f"{age}"

    return df_join.reset_index()


In [ ]:
# get transcripts 
transcripts = tbdb.getTranscripts({"corpusName" : "childes", 'corpora':[['childes', 'Eng-NA']], "age": [{'from': 24, 'to': 24}]})
df_24 = pd.DataFrame(data = transcripts['data'], columns= transcripts['colHeadings'])
df_24

In [ ]:
files_24 = pd.read_csv('CSVS/Childes_Transcripts_List/childes_transcripts_24.csv', header = None)
events_df = get_events_through_files(files_df = files_24, age = "24")

In [ ]:
for i, events in enumerate(events_df['Event_Onsets']): 
    df.loc[i, "Frequency"] = len(events)
    b_value = calc_burstiness(events = events, alternative=True)
    events_df.loc[i,"Burstiness"] = b_value


In [ ]:
# get averaged values across transcripts 
df = events_df
df.loc[df['Frequency'] == 2, 'Burstiness'] = np.nan
df = df[["Word", "Burstiness", "Frequency"]]
df_avg = df.groupby(by = "Word").mean()
df_avg.rename(columns={'Frequency': 'Mean_Frequency'}, inplace=True)
df_avg.rename(columns={'Burstiness': 'Mean_Burstiness'}, inplace=True)
#df = df.set_index(keys = "Term")
df = df[["Word", "Frequency"]]
df_sum = df.groupby(by = 'Word').sum()
df_sum.rename(columns={'Frequency': 'Sum_Frequency'}, inplace=True)

df_join = df_avg.join(df_sum)
df_join = df_join[df_join["Mean_Frequency"] != 0]
df_join["Log_Mean_Frequency"] = [math.log(f) for f in df_join["Mean_Frequency"]]
#df_join = df_join.dropna()
df_join.head()
df_join

In [ ]:
# add wordbank info to burstiness csv 
df_bvalues = df_join
df_test = df_bvalues.join(mcdi_items_df, how="left")
df_test

In [ ]:
df_test.to_csv("CSVS/b_values_childes_24.csv", index = False)